# 12 — Diagnóstico de falsos positivos

**Actividad de Grado MIA UC**  
**Sistema Inteligente de Vigilancia de Riesgos basado en RAG y LLMs**

## Objetivo
Analizar los falsos positivos producidos por la primera extracción de riesgos, establecer el baseline de **64,6 % de precisión** y clasificar las causas de error que orientarán el agente validador.

> Este notebook no modifica la extracción original. La revisión manual es la fuente de verdad.

## 1. Preparación

Ejecuta el notebook desde la raíz del repositorio o deja que la celda siguiente la detecte automáticamente.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('No se encontró la raíz del repositorio.')

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))
from src.evaluation.analyze_false_positives import (
    ERROR_TYPES, build_summaries, export_results, prepare_diagnostic, read_table
)

OUTPUT_DIR = REPO_ROOT / 'data/evaluation/risk_validation/day_01'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Raíz:', REPO_ROOT)
print('Salidas:', OUTPUT_DIR)

## 2. Seleccionar el archivo completo

Copia el archivo con los **96 riesgos evaluados** en `data/evaluation/risk_validation/` y cambia `INPUT_FILE` si su nombre es diferente.

El archivo debe conservar la columna `riesgo_valido_manual`: **1 = válido**, **0 = falso positivo**.

In [ ]:
INPUT_FILE = REPO_ROOT / 'data/evaluation/risk_validation/riesgos_evaluation_completa.xlsx'
FALLBACK_FILE = REPO_ROOT / 'data/evaluation/timeline/riesgos_priorizados.csv'

if INPUT_FILE.exists():
    source_path = INPUT_FILE
    print('Modo diagnóstico completo:', source_path)
else:
    source_path = FALLBACK_FILE
    print('ADVERTENCIA: no está el archivo de 96 riesgos.')
    print('Se usa temporalmente el archivo de 62 riesgos válidos para verificar el flujo.')
    print('No es posible diagnosticar los 34 falsos positivos con este archivo.')

source_df = read_table(source_path)
print('Registros:', len(source_df))
print('Columnas:', list(source_df.columns))
display(source_df.head(3))

## 3. Preparar la plantilla de diagnóstico

La sugerencia automática ayuda a ordenar la revisión, pero no reemplaza la clasificación humana.

In [ ]:
diagnostic_df, label_col = prepare_diagnostic(source_df)
print('Columna de validez detectada:', label_col)
display(pd.DataFrame([{'tipo_error': k, 'definicion': v} for k, v in ERROR_TYPES.items()]))
display(diagnostic_df.head(5))

## 4. Calcular el baseline

Con los 96 registros esperados, el resultado debe mostrar 62 riesgos válidos, 34 falsos positivos y una precisión cercana a 0,646.

In [ ]:
summaries = build_summaries(diagnostic_df, label_col)
print(json.dumps(summaries['metrics'], ensure_ascii=False, indent=2))
for name, table in summaries.items():
    if isinstance(table, pd.DataFrame):
        print(f'\n{name}: {len(table)} registros')
        display(table.head(15))

## 5. Exportar y completar la revisión manual

Abre `plantilla_diagnostico_falsos_positivos.xlsx`, filtra `riesgo_valido_manual = 0` y completa:

- `tipo_error_manual`
- `causa_raiz_manual`
- `evidencia_suficiente_manual`
- `correccion_requerida_manual`
- `comentario_diagnostico_manual`

In [ ]:
export_results(diagnostic_df, summaries, OUTPUT_DIR)
print('Archivos generados:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print('-', path.name)

## 6. Consolidar la clasificación manual

Después de completar y guardar el Excel, ejecuta esta celda para producir la distribución definitiva de causas.

In [ ]:
REVIEWED_FILE = OUTPUT_DIR / 'plantilla_diagnostico_falsos_positivos.xlsx'
reviewed_df = read_table(REVIEWED_FILE)
reviewed_diagnostic_df, reviewed_label_col = prepare_diagnostic(reviewed_df)
reviewed_summaries = build_summaries(reviewed_diagnostic_df, reviewed_label_col)
export_results(reviewed_diagnostic_df, reviewed_summaries, OUTPUT_DIR)
print(json.dumps(reviewed_summaries['metrics'], ensure_ascii=False, indent=2))
if 'by_error_type' in reviewed_summaries:
    display(reviewed_summaries['by_error_type'])
else:
    print('Aún falta completar tipo_error_manual para los falsos positivos.')

## 7. Criterio de cierre del Día 1

El día queda terminado cuando:

1. Los 96 registros están disponibles.
2. Se reproducen 62 válidos y 34 falsos positivos.
3. Los 34 falsos positivos tienen un tipo de error manual.
4. Se identifica cuáles tres causas concentran la mayor cantidad de errores.
5. Se redactan tres reglas que deberá aplicar el validador del Día 6.

### Texto para la bitácora
La primera extracción produjo 96 candidatos, de los cuales 62 fueron validados como riesgos y 34 como falsos positivos, para una precisión de 64,6 %. El análisis de errores permitió identificar las causas predominantes y establecer criterios verificables para la etapa de validación automática.